### Import library

In [1]:
import os
os.chdir('../../')
os.getcwd()

'C:\\Users\\csia7\\OneDrive\\문서\\GitHub\\BJ46158_WQBrain_API'

In [2]:
import ace_lib as ace
import helpful_functions as hf
import pandas as pd
import requests
import plotly.express as px
import pygwalker as pyg
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

### Start session
Enter credentials once - they will be saved to local folder and loaded each time

In [3]:
s = ace.start_session()

Complete biometrics authentication and press any key to continue: 
https://api.worldquantbrain.com/authentication/persona?inquiry=inq_TiBtqVA8fP7UszQzKkoJ1v28tZUm

 


## Global region Alpha Template

In [5]:
expression_template = f'''
group = (country+1)*group_max(pv13_52_minvol_1m_all_delay_1_sector, market) + pv13_52_minvol_1m_all_delay_1_sector;
group_neutralize(ts_scale(group_backfill(fnd23_intfvalld1_ecns, industry, 252, std=1), 252), densify(group))
'''

#### Step 1. Download datsets

In [6]:
datasets_df = hf.get_datasets(s, region = 'JPN', universe = 'TOP1600') # by default we load all datasets USA TOP3000 delay 1    , region='GLB', universe='MINVOL1M'
datasets_df # DataFrame.head() shows first 5 rows of the dataframe 

,id,name,description,category,subcategory,region,delay,universe,coverage,turnover,valueScore,userCount,alphaCount,fieldCount,themes,researchPapers
0,analyst14,Estimations of Key Fundamentals,This dataset reports many items from financial...,"{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-analyst-estimates', 'name': 'A...",JPN,1,TOP1600,0.4792,None,4.0,34,144,167,[],[{'title': 'Research Paper 10: Investor Learni...
1,analyst15,Earnings forecasts,This dataset provides bottom-up forecast data ...,"{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-analyst-estimates', 'name': 'A...",JPN,1,TOP1600,0.8486,None,3.0,80,728,288,[],[]
2,analyst16,Real Time Estimates,This dataset provides real-time access to the ...,"{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-crowdsourced-estimates', 'name...",JPN,1,TOP1600,0.6375,None,5.0,9,62,71,[],[]
3,analyst39,Analyst estimates & financial ratios,This dataset provides a lot of information on ...,"{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-esg', 'name': 'ESG'}",JPN,1,TOP1600,0.7532,None,4.0,41,118,18,[],[]
4,analyst4,Analyst Estimate Data for Equity,This dataset provides details and aggregations...,"{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-analyst-estimates', 'name': 'A...",JPN,1,TOP1600,0.5812,None,4.0,48,168,112,[],[{'title': 'Research Paper 46: Time-Series and...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,risk60,Securities Lending Insight Data,"The dataset provides list of metrics, covering...","{'id': 'risk', 'name': 'Risk'}","{'id': 'risk-risk-models', 'name': 'Risk Models'}",JPN,1,TOP1600,0.9727,None,6.0,1,1,4,[],[]
96,risk70,Multi-Factor Model,The dataset publishes risk factors which can b...,"{'id': 'risk', 'name': 'Risk'}","{'id': 'risk-risk-factors', 'name': 'Risk Fact...",JPN,1,TOP1600,0.8948,None,5.0,15,51,58,[],[]
97,shortinterest3,Securities Lending Files Data,The data consists of loans of securities by le...,"{'id': 'shortinterest', 'name': 'Short Interest'}","{'id': 'shortinterest-short-sale-models', 'nam...",JPN,1,TOP1600,0.9537,None,5.0,2,2,2,[],[]
98,shortinterest6,SmartHoldings Model,This dataset is a global stock selection model...,"{'id': 'shortinterest', 'name': 'Short Interest'}","{'id': 'shortinterest-short-sale-models', 'nam...",JPN,1,TOP1600,0.8210,None,5.0,4,17,11,[],[]


In [22]:
# select needed datasets
selected_datasets_df = datasets_df[
    (datasets_df["valueScore"] > 4) &
    (datasets_df["userCount"] > 3) 
].sort_values(by=['valueScore'], ascending=False)
selected_datasets_df

,id,name,description,category,subcategory,region,delay,universe,coverage,turnover,valueScore,userCount,alphaCount,fieldCount,themes,researchPapers
83,other7,Archive News Data,"New streaming and archive data for Benzinga, M...","{'id': 'other', 'name': 'Other'}","{'id': 'other-event-data', 'name': 'Event Data'}",JPN,1,TOP1600,0.7530,None,6.0,4,7,11,[],[]
12,broker1,Broker1,No dataset description,"{'id': 'broker', 'name': 'Broker'}","{'id': 'broker-broker-estimates', 'name': 'Bro...",JPN,1,TOP1600,0.5841,None,6.0,4,8,4,[],[]
79,other455,Relationship enhanced with AI/ML,Grouping data. Generated using deep learning m...,"{'id': 'other', 'name': 'Other'}","{'id': 'other-aiml-data', 'name': 'AI/ML Data'}",JPN,1,TOP1600,0.4670,None,6.0,14,125,1125,[],[]
78,other452,Earnings Tax Data,This dataset comprehensively studies a wide ra...,"{'id': 'other', 'name': 'Other'}","{'id': 'other-dividend-models', 'name': 'Divid...",JPN,1,TOP1600,0.5283,None,6.0,4,16,36,[],[]
2,analyst16,Real Time Estimates,This dataset provides real-time access to the ...,"{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-crowdsourced-estimates', 'name...",JPN,1,TOP1600,0.6375,None,5.0,9,62,71,[],[]
58,news48,Global Media News Data,The dataset provides news and information from...,"{'id': 'news', 'name': 'News'}","{'id': 'news-news-sentiment', 'name': 'News Se...",JPN,1,TOP1600,0.4880,None,5.0,4,9,15,[],[]
96,risk70,Multi-Factor Model,The dataset publishes risk factors which can b...,"{'id': 'risk', 'name': 'Risk'}","{'id': 'risk-risk-factors', 'name': 'Risk Fact...",JPN,1,TOP1600,0.8948,None,5.0,15,51,58,[],[]
85,other78,Earnings and conference call data,Contains records of questions and answers from...,"{'id': 'other', 'name': 'Other'}","{'id': 'other-event-data', 'name': 'Event Data'}",JPN,1,TOP1600,0.5068,None,5.0,8,18,9,[],[]
81,other515,Fund Short Selling Data,The dataset provides fund-by-fund short sellin...,"{'id': 'other', 'name': 'Other'}","{'id': 'other-aiml-data', 'name': 'AI/ML Data'}",JPN,1,TOP1600,0.2803,None,5.0,6,17,24,[],[]
75,other428,Other428,No dataset description,"{'id': 'other', 'name': 'Other'}","{'id': 'other-event-data', 'name': 'Event Data'}",JPN,1,TOP1600,0.5851,None,5.0,11,47,13,[],[]


In [38]:
# select needed datasets
selected_datasets_df_group = datasets_df[
    (datasets_df["valueScore"] > 4) &
    (datasets_df["userCount"] > 3) 
].sort_values(by=['valueScore'], ascending=False)
selected_datasets_df_group

,id,name,description,category,subcategory,region,delay,universe,coverage,turnover,valueScore,userCount,alphaCount,fieldCount,themes,researchPapers
83,other7,Archive News Data,"New streaming and archive data for Benzinga, M...","{'id': 'other', 'name': 'Other'}","{'id': 'other-event-data', 'name': 'Event Data'}",JPN,1,TOP1600,0.7530,None,6.0,4,7,11,[],[]
12,broker1,Broker1,No dataset description,"{'id': 'broker', 'name': 'Broker'}","{'id': 'broker-broker-estimates', 'name': 'Bro...",JPN,1,TOP1600,0.5841,None,6.0,4,8,4,[],[]
79,other455,Relationship enhanced with AI/ML,Grouping data. Generated using deep learning m...,"{'id': 'other', 'name': 'Other'}","{'id': 'other-aiml-data', 'name': 'AI/ML Data'}",JPN,1,TOP1600,0.4670,None,6.0,14,125,1125,[],[]
78,other452,Earnings Tax Data,This dataset comprehensively studies a wide ra...,"{'id': 'other', 'name': 'Other'}","{'id': 'other-dividend-models', 'name': 'Divid...",JPN,1,TOP1600,0.5283,None,6.0,4,16,36,[],[]
2,analyst16,Real Time Estimates,This dataset provides real-time access to the ...,"{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-crowdsourced-estimates', 'name...",JPN,1,TOP1600,0.6375,None,5.0,9,62,71,[],[]
58,news48,Global Media News Data,The dataset provides news and information from...,"{'id': 'news', 'name': 'News'}","{'id': 'news-news-sentiment', 'name': 'News Se...",JPN,1,TOP1600,0.4880,None,5.0,4,9,15,[],[]
96,risk70,Multi-Factor Model,The dataset publishes risk factors which can b...,"{'id': 'risk', 'name': 'Risk'}","{'id': 'risk-risk-factors', 'name': 'Risk Fact...",JPN,1,TOP1600,0.8948,None,5.0,15,51,58,[],[]
85,other78,Earnings and conference call data,Contains records of questions and answers from...,"{'id': 'other', 'name': 'Other'}","{'id': 'other-event-data', 'name': 'Event Data'}",JPN,1,TOP1600,0.5068,None,5.0,8,18,9,[],[]
81,other515,Fund Short Selling Data,The dataset provides fund-by-fund short sellin...,"{'id': 'other', 'name': 'Other'}","{'id': 'other-aiml-data', 'name': 'AI/ML Data'}",JPN,1,TOP1600,0.2803,None,5.0,6,17,24,[],[]
75,other428,Other428,No dataset description,"{'id': 'other', 'name': 'Other'}","{'id': 'other-event-data', 'name': 'Event Data'}",JPN,1,TOP1600,0.5851,None,5.0,11,47,13,[],[]


#### Step 2. Select the needed datafields

In [23]:
dataset_ids = selected_datasets_df.id.values.tolist() # create a list of selected datasets ids, our list has only one element
dataset_ids

['other7',
 'broker1',
 'other455',
 'other452',
 'analyst16',
 'news48',
 'risk70',
 'other78',
 'other515',
 'other428',
 'other395',
 'other323',
 'news7',
 'news54',
 'news50',
 'news38',
 'analyst44',
 'news31',
 'news3',
 'news20',
 'news18',
 'news17',
 'model33',
 'model176',
 'model16',
 'model106',
 'analyst9',
 'analyst47',
 'analyst45',
 'shortinterest6']

In [24]:
selected_datafields_full = []
for dataset_id in dataset_ids:
    datafields_df = hf.get_datafields(s, region = 'JPN', universe = 'TOP1600', dataset_id=dataset_id)
    selected_datafields_df = datafields_df[
        (datafields_df['type'] == 'MATRIX')
    ].sort_values(by=['userCount'], ascending=False)
    selected_datafields_df_id = selected_datafields_df.id.values.tolist()
    selected_datafields_full += selected_datafields_df_id[:10]
selected_datafields_full

['oth455_relation_roam_w5_pca_fact3_value',
 'oth455_relation_roam_w4_pca_fact2_value',
 'oth455_partner_roam_w1_pca_fact1_value',
 'oth455_relation_n2v_p10_q50_w2_pca_fact2_value',
 'oth455_relation_n2v_p10_q50_w5_pca_fact3_value',
 'oth455_customer_n2v_p10_q50_w5_pca_fact1_value',
 'oth455_customer_n2v_p10_q200_w1_pca_fact3_value',
 'oth455_relation_n2v_p10_q200_w5_pca_fact3_value',
 'oth455_relation_n2v_p50_q50_w2_pca_fact2_value',
 'oth455_customer_n2v_p50_q200_w1_pca_fact3_value',
 'oth452_accruals_d1_min',
 'oth452_liability_d1_length',
 'oth452_accruals_d1_length',
 'oth452_deferred_tax_expense_51',
 'oth452_liability_d1_asset_change',
 'oth452_expense_d1_asset_change',
 'oth452_deferred_tax_liability_51',
 'oth452_deferred_tax_liability_41',
 'oth452_expense_d1_mean',
 'oth452_deferred_tax_liability_13',
 'rsk70_mfm2_asetrd_season',
 'rsk70_mfm2_asetrd_dsrt',
 'rsk70_mfm2_asetrd_shortint',
 'rsk70_mfm2_asetrd_anlystsn',
 'rsk70_mfm2_asetrd_cntadj',
 'rsk70_mfm2_asetrd_momentum'

In [40]:
dataset_group_ids = selected_datasets_df_group.id.values.tolist() # create a list of selected datasets ids, our list has only one element
dataset_group_ids

['other7',
 'broker1',
 'other455',
 'other452',
 'analyst16',
 'news48',
 'risk70',
 'other78',
 'other515',
 'other428',
 'other395',
 'other323',
 'news7',
 'news54',
 'news50',
 'news38',
 'analyst44',
 'news31',
 'news3',
 'news20',
 'news18',
 'news17',
 'model33',
 'model176',
 'model16',
 'model106',
 'analyst9',
 'analyst47',
 'analyst45',
 'shortinterest6']

In [41]:
selected_datafields_group_full = []
for dataset_id in dataset_group_ids:
    datafields_group_df = hf.get_datafields(s, region = 'JPN', universe = 'TOP1600', dataset_id=dataset_id)
    selected_datafields_group_df = datafields_group_df[
        (datafields_group_df['type'] == 'GROUP')
    ].sort_values(by=['userCount'], ascending=False)
    selected_datafields_group_df_id = selected_datafields_group_df.id.values.tolist()
    selected_datafields_group_full += selected_datafields_group_df_id[:10]
selected_datafields_group_full

['oth455_relation_n2v_p10_q50_w2_kmeans_cluster_20',
 'oth455_partner_roam_w5_pca_fact2_cluster_10',
 'oth455_customer_n2v_p10_q200_w1_pca_fact3_cluster_5',
 'oth455_partner_n2v_p10_q200_w2_kmeans_cluster_20',
 'oth455_partner_n2v_p10_q50_w2_pca_fact2_cluster_20',
 'oth455_customer_roam_w3_pca_fact3_cluster_10',
 'oth455_relation_n2v_p10_q50_w5_pca_fact2_cluster_10',
 'oth455_customer_n2v_p50_q50_w4_pca_fact3_cluster_20',
 'oth455_customer_roam_w2_pca_fact1_cluster_10',
 'oth455_partner_n2v_p50_q50_w4_kmeans_cluster_10']

#### Step 4. Apply generate_alpha function to the expression list
In generate alpha function you can specify region, universe, decay, delay and other simulation settings

이제 만들어둔 expressions 을 사용해 alpha simulation 을 해줍니다.

In [61]:
#?ace.generate_alpha

In [32]:

expression_template = f'''
group = (country+1)*group_max({grp}, market) + {grp};
group_neutralize(ts_scale(group_backfill(rsk70_mfm2_asetrd_season, industry, 252, std=1), 252), densify(group))
'''

In [42]:
expressions = []
for grp in selected_datafields_group_full:
    expressions.append(f'group = (country+1)*group_max({grp}, market) + {grp};group_neutralize(ts_scale(group_backfill(rsk70_mfm2_asetrd_season, industry, 252, std=1), 252), densify(group))')

In [43]:
len(expressions)

10

In [44]:
#when you send multiple alphas for simulation, please make sure all alphas of a single list should have common settings
#alphas with different settings should be sent in a different list, for instance below list has all alphas with same settings

alpha_list = [ace.generate_alpha(x, region= "JPN", universe = "TOP1600", truncation = 0.01, delay = 1, decay = 3) for x in expressions]
alpha_list[0]

{'type': 'REGULAR',
 'settings': {'instrumentType': 'EQUITY',
  'region': 'JPN',
  'universe': 'TOP1600',
  'delay': 1,
  'decay': 3,
  'neutralization': 'INDUSTRY',
  'truncation': 0.01,
  'pasteurization': 'ON',
  'testPeriod': 'P0Y0M0D',
  'unitHandling': 'VERIFY',
  'nanHandling': 'OFF',
  'language': 'FASTEXPR',
  'visualization': False},
 'regular': 'group = (country+1)*group_max(oth455_relation_n2v_p10_q50_w2_kmeans_cluster_20, market) + oth455_relation_n2v_p10_q50_w2_kmeans_cluster_20;group_neutralize(ts_scale(group_backfill(rsk70_mfm2_asetrd_season, industry, 252, std=1), 252), densify(group))'}

### Simulate alpha list, get simulation result

simulate_alpha_list_multi will do a multi-simulation if list of alphas is greater than 10, which is the case here

the returned object will contain simulation results for all alphas as a list

In [45]:
#alpha expressions are sliced to first 10 for demonstration purpose

result = ace.simulate_alpha_list_multi(s, alpha_list)

100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [04:51<00:00, 72.92s/it]


위 코드를 실행하면 시뮬레이션이 시작됩니다. 100개의 알파에 대략 1시간 정도 소요되니, 인터넷 연결이 끊기지 않게 주의하시고, 너무 많은 알파를 한 번에 돌리기보다는 몇개씩 끊어서 돌려도 좋습니다.

In [46]:
#prettify_result function can be used from the helpful_functions library to take a look at IS stats of all the simulated alphas

result_st1 = hf.prettify_result(result, detailed_tests_view=False)
result_st1

,pnl,book_size,long_count,short_count,turnover,returns,drawdown,margin,fitness,sharpe,start_date,alpha_id,expression,concentrated_weight,high_turnover,is_ladder_sharpe,low_fitness,low_sharpe,low_sub_universe_sharpe,low_turnover
0,5257119,20000000,615,614,0.2534,0.0508,0.1483,0.000401,0.77,1.72,2012-01-22,rlXL5EJ,group = (country+1)*group_max(oth455_relation_...,PASS,PASS,PASS,FAIL,PASS,FAIL,PASS
1,5118475,20000000,615,614,0.2536,0.0494,0.1480,0.000390,0.75,1.71,2012-01-22,3RLx93z,group = (country+1)*group_max(oth455_relation_...,PASS,PASS,PASS,FAIL,PASS,FAIL,PASS
2,4796056,20000000,595,593,0.2528,0.0463,0.1381,0.000366,0.66,1.55,2012-01-22,2LdROQY,group = (country+1)*group_max(oth455_partner_r...,PASS,PASS,PASS,FAIL,FAIL,FAIL,PASS
3,4657658,20000000,595,593,0.2539,0.0450,0.1426,0.000354,0.64,1.51,2012-01-22,APKb0bW,group = (country+1)*group_max(oth455_partner_n...,PASS,PASS,FAIL,FAIL,FAIL,FAIL,PASS
4,4658416,20000000,595,593,0.2524,0.0450,0.1419,0.000356,0.64,1.52,2012-01-22,anvlxGW,group = (country+1)*group_max(oth455_partner_n...,PASS,PASS,PASS,FAIL,FAIL,FAIL,PASS
5,4546217,20000000,595,593,0.2528,0.0439,0.1447,0.000347,0.62,1.49,2012-01-22,K9a6OqE,group = (country+1)*group_max(oth455_partner_n...,PASS,PASS,PASS,FAIL,FAIL,FAIL,PASS
6,3160180,20000000,475,474,0.2517,0.0305,0.1444,0.000242,0.31,0.88,2012-01-22,bqEMRMm,group = (country+1)*group_max(oth455_customer_...,PASS,PASS,FAIL,FAIL,FAIL,FAIL,PASS
7,3091945,20000000,475,474,0.2510,0.0299,0.1427,0.000238,0.29,0.84,2012-01-22,5OW9Q9z,group = (country+1)*group_max(oth455_customer_...,PASS,PASS,FAIL,FAIL,FAIL,FAIL,PASS
8,2819425,20000000,475,474,0.2521,0.0272,0.1378,0.000216,0.25,0.77,2012-01-22,8Q1r3rz,group = (country+1)*group_max(oth455_customer_...,PASS,PASS,FAIL,FAIL,FAIL,FAIL,PASS
9,2636785,20000000,475,474,0.2535,0.0255,0.1397,0.000201,0.23,0.72,2012-01-22,Xn6LWPz,group = (country+1)*group_max(oth455_customer_...,PASS,PASS,FAIL,FAIL,FAIL,FAIL,PASS


In [37]:
result_st1['expression'][0]

'group_neutralize(ts_scale(group_backfill(rsk70_mfm2_asetrd_season, industry, 252, std=1), 252), densify(country))'

prettify 함수를 이용해 dataframe 형식으로 변환하면 결과를 보기 더 편합니다.

In [30]:
result_st1.to_csv('2024_09_018_JPN_2.csv')

### How to submit?

Create a list of submittable alphas - alphas that have no FAIL in is_tests

시뮬레이션을 마친 알파 중 제출 가능한 알파들을 제출해 봅시다.

In [47]:
#to take a look at the combined result of all new alphas

is_tests_df = hf.concat_is_tests(result)
is_tests_df

,alpha_id,endDate,limit,message,name,result,startDate,value,year
0,rlXL5EJ,NaN,1.58,NaN,LOW_SHARPE,PASS,NaN,1.7200,NaN
1,rlXL5EJ,NaN,1.00,NaN,LOW_FITNESS,FAIL,NaN,0.7700,NaN
2,rlXL5EJ,NaN,0.01,NaN,LOW_TURNOVER,PASS,NaN,0.2534,NaN
3,rlXL5EJ,NaN,0.70,NaN,HIGH_TURNOVER,PASS,NaN,0.2534,NaN
4,rlXL5EJ,NaN,NaN,NaN,CONCENTRATED_WEIGHT,PASS,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
115,anvlxGW,NaN,NaN,NaN,SELF_CORRELATION,PENDING,NaN,NaN,NaN
116,anvlxGW,NaN,NaN,NaN,DATA_DIVERSITY,PENDING,NaN,NaN,NaN
117,anvlxGW,NaN,NaN,NaN,PROD_CORRELATION,PENDING,NaN,NaN,NaN
118,anvlxGW,NaN,NaN,NaN,REGULAR_SUBMISSION,PENDING,NaN,NaN,NaN


In [48]:
#making a list of failed alphas
failed_alphas = is_tests_df.query('result=="FAIL"')['alpha_id'].unique()

#making a list of passed alphas
passed_alphas = list(set(is_tests_df['alpha_id']).difference(failed_alphas))

print(f'Failed alphas:{failed_alphas}\nPassed alphas:{passed_alphas}')

Failed alphas:['rlXL5EJ' 'Xn6LWPz' '8Q1r3rz' 'K9a6OqE' 'APKb0bW' 'bqEMRMm' '3RLx93z'
 '2LdROQY' '5OW9Q9z' 'anvlxGW']
Passed alphas:[]


In [5]:
for alpha_id in ['ad59v2O', 'k3z6qqk', 'Lp0YPva', '9wNOzNr', '7weR901', '6w5Ln6E', 'QawpLe5', '17rAQ6m', '273obEY', 'xxJpKLg', 'Wpv2EZo', 'm8EWMk1']:
    hf.set_alpha_properties(s, alpha_id, tags = ['pass_Sep07_mixing'])

In [27]:
for alpha_id in passed_alphas:
    hf.set_alpha_properties(s, alpha_id, tags = ['Sep07_GLB'])

위 함수로 통과한 알파들에 태그를 붙일 수 있습니다.

When you got a list of submittable alphas, you can call function submit_alpha()

In [50]:
#calling submit_alpha on all alphas that have passed the submission tests

submit_result = {alpha_id: ace.submit_alpha(s, alpha_id) for alpha_id in passed_alphas}

위 함수로 알파를 제출할 수 있습니다.

알파의 id 로 제출하는 것이기 때문에, 그 알파를 시뮬레이션 한 날짜가 중요합니다. 가령, 1주일 전에 시뮬레이션 해 결과를 저장해 두었던 alpha id로 제출을 한다면 제출 날짜가 1주일 전이 됩니다. 그러니 오늘 제출하고 싶은 alpha 가 있다면 시뮬레이션을 다시 돌려 새로 생긴 alpha id 를 이용해야 합니다. 이 과정은 alpha 에 특정 태그를 달아 Brain 플랫폼에서 직접 하는게 훨씬 수월합니다. (다만, 플랫폼에서는 alpha id 로 알파를 검색할 수 없습니다.)

In [51]:
#submit_result will have return values from the submit_result function

submit_result

{'ZpO36rQ': False}

주의하셔야 하실 점은, 하루 최대 알파 제출 한도인 4개를 루프중에 이미 넘기면 그 뒤의 알파들이 제출 가능하더라도 자동으로 제출에 실패하게 됩니다.

### Library Fuctions.

following are some other functions that you can use for your own analysis

**get_alpha_pnl(s, alpha_id)** - to get the pnl for an alpha

**get_alpha_yearly_stats(s, alpha_id)** - to get yearly statistics for an alpha

**get_self_corr(s, alpha_id)** - to get self correlation results for an alpha

**get_prod_corr(s, alpha_id)** - to get prod correlation results for an alpha

**get_check_submission(s, alpha_id)** - to get check submission result for an alpha

**check_self_corr_test(s, alpha_id)** - to check if alpha passes self correlation test (self_corr<0.7)

**check_prod_corr_test(s, alpha_id)** - to check if alpha passes prod correlation test (prod_corr<0.7)

**perfomance_comparison(s, alpha_id)** - to get the result of performance comparison for an alpha merged performance

유용한 추가 함수들입니다. prod correlation 혹은 self correlation 만을 확인하고 싶을 때, 혹은 알파의 pnl 을 확인하고 싶을때 유용히 쓰입니다.